In [71]:
import os
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.utils.data import random_split


In [72]:
batch_size = 128

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)


CNN amélioré

In [73]:
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32x32 -> 16x16
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16x16 -> 8x8
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 8x8 -> 4x4
            nn.Dropout(0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

Entraînement

In [74]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ImprovedCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()*images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss/total, correct/total

def test_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()*images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss/total, correct/total

In [75]:
class ImprovedCNN_MCdropout(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        # on force le dropout pendant l'inférence
        def apply_mc_dropout(module):
            if isinstance(module, nn.Dropout) or isinstance(module, nn.Dropout2d):
                module.train()
        
        self.base_model.apply(apply_mc_dropout)
        return self.base_model(x)

In [76]:
class ImprovedCNN_MCdropout_generic(nn.Module):
    def __init__(self, dico_layers):
        super().__init__()
        self.base_model = dico_layers['model']
        self.mc_layers = dico_layers.get('layers', {})

    def forward(self, x):
        def apply_mc_dropout(module):
            if isinstance(module, nn.Dropout) or isinstance(module, nn.Dropout2d):
                # On applique le dropout seulement si on a choisi cette couche
                # On peut mettre un mapping mc_layers[nom] si on veut par couche
                module.train()
        
        # Appliquer MC Dropout sur tout le modèle
        self.base_model.apply(apply_mc_dropout)
        return self.base_model(x)

Boucle principale

In [77]:
def train_model(model, trainloader, valloader, device, epochs=20, save_path="best.pt"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0
    best_test_acc = 0
    base_model = ImprovedCNN().to(device)

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_epoch(model, trainloader, optimizer, criterion, device)
        test_loss, test_acc = test_epoch(model, testloader, criterion, device)
        scheduler.step()

    # Sauvegarde si meilleur test accuracy
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            torch.save(base_model.state_dict(), "best_improved.pt")
            print(f"→ Meilleur modèle sauvegardé à l'epoch {epoch+1} | Test Acc: {test_acc:.4f}")

        if (epoch+1) % 5 == 0 or epoch==0:
            print(f"Epoch {epoch+1}/{epochs} | "
                f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.4f} | "
                f"Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")

In [78]:
def evaluate(model, dataloader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss/len(dataloader), correct/total

In [79]:
# base_model = ImprovedCNN().to(device)

# if os.path.exists("best_improved.pt"):
#     base_model.load_state_dict(torch.load("best_improved.pt", map_location=device))
# else:
#     # Sinon, entraîner le modèle et sauvegarder les poids
#     train_model(base_model, trainloader, testloader, optimizer, criterion, scheduler, device)

# # Créer le modèle MC Dropout à partir des meilleurs poids
# model = ImprovedCNN_MCdropout(copy.deepcopy(base_model)).to(device)

# # Évaluation finale
# test_loss, test_acc = evaluate(model, testloader, device)
# print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")

In [80]:
val_ratio = 0.1  # 10% pour validation
train_size = int((1 - val_ratio) * len(trainset))
val_size   = len(trainset) - train_size

train_subset, val_subset = random_split(trainset, [train_size, val_size]) # divise le dataset de manière aléatoire en 90/10

trainloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader   = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

In [81]:
def evaluate(model, dataloader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss/len(dataloader), correct/total

In [82]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best_improved.pt"

# Demande à l'utilisateur quelles couches doivent subir le dropout
user_layers = input(
    "Sur quelles couches voulez-vous appliquer le MC Dropout ? "
    "(choisissez parmi features1, features2, features3, classifier1, séparées par des virgules) : ")
mc_layers = [layer.strip() for layer in user_layers.split(',')]

# Vérifie si les poids existent déjà
base_model = ImprovedCNN().to(device)
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, testloader, optimizer, criterion, scheduler, device)

# Dictionnaire pour CNN_MCdropout_generic
dico_layers = {
    "model": base_model,
    "layers": {layer: 0.1 for layer in mc_layers}  # probabilités arbitraires
}

# Copie pour MC Dropout
base_model_mc = copy.deepcopy(base_model)

# Création des modèles MC Dropout
model = ImprovedCNN_MCdropout(base_model_mc).to(device)
model2 = ImprovedCNN_MCdropout_generic(dico_layers).to(device)

# Évaluation finale
test_loss, test_acc = evaluate(model, testloader, device)
print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")

test_loss2, test_acc2 = evaluate(model2, testloader, device)
print(f"Final Test Loss 2: {test_loss2:.4f} - Test Acc 2: {test_acc2:.4f}")


Chargement du modèle sauvegardé
Final Test Loss: 2.3034 - Test Acc: 0.1008
Final Test Loss 2: 2.3036 - Test Acc 2: 0.0982
